## 0- inspeção dos dados

Primeira coisa que iremos fazer é verificar como estão os dados no dataset. 
então a primeira coisa é verificar se tem nulos, duplicatas e se os tipos de dados estão corretos 

In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/Consumo_cerveja.csv")

In [2]:
df.shape

(941, 7)

In [3]:
df.head()

,Data,Temperatura Media (C),Temperatura Minima (C),Temperatura Maxima (C),Precipitacao (mm),Final de Semana,Consumo de cerveja (litros)
0,2015-01-01,"27,3","23,9","32,5",0,0.0,25.461
1,2015-01-02,"27,02","24,5","33,5",0,0.0,28.972
2,2015-01-03,"24,82","22,4","29,9",0,1.0,30.814
3,2015-01-04,"23,98","21,5","28,6","1,2",1.0,29.799
4,2015-01-05,"23,82",21,"28,3",0,0.0,28.900


In [4]:
df.tail()

,Data,Temperatura Media (C),Temperatura Minima (C),Temperatura Maxima (C),Precipitacao (mm),Final de Semana,Consumo de cerveja (litros)
936,NaN,NaN,NaN,NaN,NaN,NaN,NaN
937,NaN,NaN,NaN,NaN,NaN,NaN,NaN
938,NaN,NaN,NaN,NaN,NaN,NaN,NaN
939,NaN,NaN,NaN,NaN,NaN,NaN,NaN
940,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 941 entries, 0 to 940
Data columns (total 7 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Data                         365 non-null    str    
 1   Temperatura Media (C)        365 non-null    str    
 2   Temperatura Minima (C)       365 non-null    str    
 3   Temperatura Maxima (C)       365 non-null    str    
 4   Precipitacao (mm)            365 non-null    str    
 5   Final de Semana              365 non-null    float64
 6   Consumo de cerveja (litros)  365 non-null    float64
dtypes: float64(2), str(5)
memory usage: 51.6 KB


In [6]:
df.describe()

,Final de Semana,Consumo de cerveja (litros)
count,365.000000,365.000000
mean,0.284932,25.401367
std,0.452001,4.399143
min,0.000000,14.343000
25%,0.000000,22.008000
50%,0.000000,24.867000
75%,1.000000,28.631000
max,1.000000,37.937000


In [7]:
df.isnull().sum()

Data                           576
Temperatura Media (C)          576
Temperatura Minima (C)         576
Temperatura Maxima (C)         576
Precipitacao (mm)              576
Final de Semana                576
Consumo de cerveja (litros)    576
dtype: int64

In [8]:
df.duplicated().sum()

np.int64(575)

In [9]:
df_limpo = df.dropna(how="all").copy()
df_limpo.shape

(365, 7)

In [10]:
df_limpo.duplicated().sum()

np.int64(0)

In [11]:
df_limpo.isnull().sum()

Data                           0
Temperatura Media (C)          0
Temperatura Minima (C)         0
Temperatura Maxima (C)         0
Precipitacao (mm)              0
Final de Semana                0
Consumo de cerveja (litros)    0
dtype: int64

## Conclusão da inspeção: 

1- Verificamos que nos dados existiam 576 linhas totalmente vazias e então excluimos elas.

2- Temperaturas e chuva como texto.

3- data como texto.

4- final de semana está como float mas na verdade deve ser inteiro 0 ou 1.

5- nomes de colunas com espaços e colunas.

## 1- Limpeza e engenharia de dados 

agora vamos limpar e transformar os dados para que a analise não tenha erros.

In [12]:
# renomenado as colunas para nomes menos complexos e sem espaços para não ter problemas futuros 

df_limpo = df_limpo.rename(columns={
    "Data": "data",
    "Temperatura Media (C)": "temp_media",
    "Temperatura Minima (C)": "temp_min",
    "Temperatura Maxima (C)": "temp_max",
    "Precipitacao (mm)": "chuva_mm",
    "Final de Semana": "fim_de_semana",
    "Consumo de cerveja (litros)": "consumo_mil_litros",
})

In [13]:
# substituindo tudo que for "," por "." nas 4 colunas indicadas

colunas_com_virgula = ["temp_media", "temp_min", "temp_max", "chuva_mm"]

for coluna in colunas_com_virgula:
    df_limpo[coluna] = df_limpo[coluna].str.replace(",", ".").astype(float)

In [14]:
# transformando a coluna data em datetime

df_limpo["data"] = pd.to_datetime(df_limpo["data"], format = "%Y-%m-%d")

In [15]:
# converter o fim de semana para inteiro 
df_limpo["fim_de_semana"] = df_limpo["fim_de_semana"].astype(int)

In [16]:
df_limpo.info()

<class 'pandas.DataFrame'>
RangeIndex: 365 entries, 0 to 364
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   data                365 non-null    datetime64[us]
 1   temp_media          365 non-null    float64       
 2   temp_min            365 non-null    float64       
 3   temp_max            365 non-null    float64       
 4   chuva_mm            365 non-null    float64       
 5   fim_de_semana       365 non-null    int64         
 6   consumo_mil_litros  365 non-null    float64       
dtypes: datetime64[us](1), float64(5), int64(1)
memory usage: 20.1 KB


In [17]:
df_limpo.describe()

,data,temp_media,temp_min,temp_max,chuva_mm,fim_de_semana,consumo_mil_litros
count,365,365.000000,365.000000,365.000000,365.000000,365.000000,365.000000
mean,2015-07-02 00:00:00,21.226356,17.461370,26.611507,5.196712,0.284932,25.401367
min,2015-01-01 00:00:00,12.900000,10.600000,14.500000,0.000000,0.000000,14.343000
25%,2015-04-02 00:00:00,19.020000,15.300000,23.800000,0.000000,0.000000,22.008000
50%,2015-07-02 00:00:00,21.380000,17.900000,26.900000,0.000000,0.000000,24.867000
75%,2015-10-01 00:00:00,23.280000,19.600000,29.400000,3.200000,1.000000,28.631000
max,2015-12-31 00:00:00,28.860000,24.500000,36.500000,94.800000,1.000000,37.937000
std,NaN,3.180108,2.826185,4.317366,12.417844,0.452001,4.399143


In [18]:
df_limpo.head()

,data,temp_media,temp_min,temp_max,chuva_mm,fim_de_semana,consumo_mil_litros
0,2015-01-01,27.30,23.9,32.5,0.0,0,25.461
1,2015-01-02,27.02,24.5,33.5,0.0,0,28.972
2,2015-01-03,24.82,22.4,29.9,0.0,1,30.814
3,2015-01-04,23.98,21.5,28.6,1.2,1,29.799
4,2015-01-05,23.82,21.0,28.3,0.0,0,28.900


In [19]:
# criação de duas colunas novas para calculos futuros 

df_limpo["mes"] = df_limpo["data"].dt.month
df_limpo["dia_semana_num"] = df_limpo["data"].dt.dayofweek

nomes_dias = {
    0 : "segunda", 1 : "terça", 2 : "quarta", 3 : "quinta",
    4 : "sexta", 5 : "sábado", 6 : "domingo",
}

df_limpo["dia_semana"] = df_limpo["dia_semana_num"].map(nomes_dias)

In [20]:
df_limpo.head()

,data,temp_media,temp_min,temp_max,chuva_mm,fim_de_semana,consumo_mil_litros,mes,dia_semana_num,dia_semana
0,2015-01-01,27.30,23.9,32.5,0.0,0,25.461,1,3,quinta
1,2015-01-02,27.02,24.5,33.5,0.0,0,28.972,1,4,sexta
2,2015-01-03,24.82,22.4,29.9,0.0,1,30.814,1,5,sábado
3,2015-01-04,23.98,21.5,28.6,1.2,1,29.799,1,6,domingo
4,2015-01-05,23.82,21.0,28.3,0.0,0,28.900,1,0,segunda


In [21]:
fim_calculado = (df_limpo["dia_semana_num"] >= 5).astype(int)
(fim_calculado != df_limpo["fim_de_semana"]).sum()

np.int64(0)

In [22]:
df_limpo.info()

<class 'pandas.DataFrame'>
RangeIndex: 365 entries, 0 to 364
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   data                365 non-null    datetime64[us]
 1   temp_media          365 non-null    float64       
 2   temp_min            365 non-null    float64       
 3   temp_max            365 non-null    float64       
 4   chuva_mm            365 non-null    float64       
 5   fim_de_semana       365 non-null    int64         
 6   consumo_mil_litros  365 non-null    float64       
 7   mes                 365 non-null    int32         
 8   dia_semana_num      365 non-null    int32         
 9   dia_semana          365 non-null    str           
dtypes: datetime64[us](1), float64(5), int32(2), int64(1), str(1)
memory usage: 25.8 KB


In [23]:
df_limpo.to_csv("../data/processed/consumo_sao_paulo.csv", index=False)

## Conclusão da limpeza:

Agora está tudo organizado para as devidas analises, convertemos colunas que tinha dados com formatação errada como a temperatura e data.
também criamos duas colunas adicionais que é mes e o dia da semana para ajudar nas analises temporais.